In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""
    SELECT 
        id AS repo_id,
        LOWER(name) AS name,
        full_name,
        org_login,
        description,
        language,
        license_name,
        created_at
    FROM silver.{repo_name}_repo_data
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ALL ".join(union_parts)
full_query = f"""
CREATE OR REPLACE VIEW gold.dim_repo AS
WITH combined_repos AS (
    {combined_subquery}
)
SELECT 
    ROW_NUMBER() OVER(ORDER BY repo_id) AS repo_key,
    repo_id,
    name,
    full_name,
    org_login,
    description,
    language,
    license_name,
    created_at
FROM combined_repos
"""
spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.dim_repo

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""
    SELECT 
        DISTINCT author_association
    FROM silver.{repo_name}_pull_requests
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
full_query = f"""
CREATE OR REPLACE VIEW gold.dim_author_association AS
WITH combined_repos AS (
    {combined_subquery}
)
SELECT 
    ROW_NUMBER() OVER(ORDER BY author_association) AS author_association_key,
    author_association
FROM combined_repos
"""
spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.dim_author_association

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT 
        state,
        CASE
            WHEN draft = true THEN 'Draft'
            ELSE 'Not Draft' 
        END AS draft_label,
        CASE
            WHEN locked = true THEN 'Locked'
            ELSE 'Not Locked' 
        END AS locked_label,
        auto_merge_method
        FROM silver.{repo_name}_pull_requests
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE VIEW gold.dim_flags
    AS
    WITH combined_repos AS (
        {combined_subquery}
    )
    SELECT DISTINCT
        ROW_NUMBER() OVER(ORDER BY state, draft_label, locked_label, auto_merge_method) AS flag_key,
        state,
        draft_label,
        locked_label,
        auto_merge_method
    FROM combined_repos    
"""
spark.sql(full_query)

In [0]:
%sql
SELECT DISTINCT state, draft, locked, auto_merge_method FROM silver.aspnetcore_pull_requests

In [0]:
%sql
SELECT * FROM gold.dim_flags

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_date AS
WITH generated_date AS (
    SELECT 
        EXPLODE(
            SEQUENCE(
                DATE '2000-01-01',
                DATE '2030-01-01',
                INTERVAL 1 DAY
            )
        ) AS full_date
)
SELECT 
    CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date AS date,
    YEAR(full_date) AS year,
    MONTH(full_date) AS month,
    DAY(full_date) AS day,
    date_format(full_date, 'EEEE') AS day_name,
    date_format(full_date, 'MMMM') AS month_name,
    QUARTER(full_date) AS quarter
FROM generated_date

In [0]:
%sql
SELECT * FROM gold.dim_date

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_timestamp AS

WITH minutes_sequence AS (
    SELECT
        explode(
            sequence(0, 1439, 1)  
        ) AS minute_of_day
)

SELECT
    minute_of_day                          AS timestamp_id,  
    CAST(minute_of_day / 60 AS INT)        AS hour,            
    CAST(minute_of_day % 60 AS INT)        AS minute           
FROM minutes_sequence
ORDER BY timestamp_id

In [0]:
%sql
SELECT * FROM gold.dim_timestamp

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT DISTINCT
            milestone_id,
            milestone_title
        FROM silver.{repo_name}_pull_requests
        WHERE milestone_id IS NOT NULL

    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE VIEW gold.dim_milestone AS
    WITH combined_repos AS (
        {combined_subquery}
    ), 
    resolved AS (
            SELECT milestone_id, MAX(milestone_title) AS milestone_title
            FROM combined_repos
            GROUP BY milestone_id
        ),
        distinct_notnull AS (
            SELECT 
                ROW_NUMBER() OVER(ORDER BY milestone_id) AS milestone_key,
                milestone_id,
                milestone_title
            FROM resolved
        ),
    null_milestone AS (
        SELECT 
            -1 AS milestone_key,
            CAST(NULL AS BIGINT) AS milestone_id,
            'n/a' AS milestone_title
    )
    SELECT * FROM null_milestone
    UNION ALL
    SELECT * FROM distinct_notnull
"""
spark.sql(full_query)

union_parts = [
    f"""
        SELECT 
            milestone_id,
            milestone_state,
            transformed_at AS date_created
        FROM silver.{repo_name}_pull_requests
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ALL ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE VIEW gold.dim_milestone_state_history
    AS
    WITH combined_repos AS (
        {combined_subquery}
        ),
        distinct_states AS (
            SELECT
                milestone_id,
                milestone_state,
                MIN(date_created) AS date_created
            FROM combined_repos
            WHERE milestone_id IS NOT NULL
            GROUP BY milestone_id, milestone_state
        ),
        null_milestone AS (
            SELECT 
                -1 AS milestone_state_key,
                -1 AS change_sequence,
                CAST(NULL AS BIGINT) AS milestone_id,
                'n/a' AS milestone_state, 
                MAX(date_created) AS date_created
            FROM combined_repos
            WHERE milestone_id IS NULL
        )
        SELECT 
            ROW_NUMBER() OVER (ORDER BY milestone_id, milestone_state) AS milestone_state_key,
            ROW_NUMBER() OVER (PARTITION BY milestone_id ORDER BY date_created) AS change_sequence,
            milestone_id,
            milestone_state,
            date_created
        FROM distinct_states
        UNION ALL
        SELECT * FROM null_milestone
    """
spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.dim_milestone ORDER BY milestone_key

In [0]:
%sql
SELECT milestone_id, COUNT(*) AS cnt
FROM gold.dim_milestone
WHERE milestone_id IS NOT NULL
GROUP BY milestone_id
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT * FROM gold.dim_milestone_state_history ORDER BY milestone_state_key

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT 
            DISTINCT
            base_ref,
            base_repo_id
        FROM silver.{repo_name}_pull_requests
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE VIEW gold.dim_base_ref
    AS
    WITH combined_repos AS (
        {combined_subquery}
    ),
        resolved AS (
            SELECT base_ref, base_repo_id
            FROM combined_repos
            GROUP BY base_ref, base_repo_id
        )
        SELECT
            ROW_NUMBER() OVER(ORDER BY base_ref, base_repo_id) AS base_ref_key,
            base_ref,
            base_repo_id
        FROM resolved    
"""
spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.dim_base_ref WHERE base_repo_id IS NULL

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT DISTINCT
            assignee_id AS user_id,
            assignee_login AS user_login,
            assignee_type AS user_type
        FROM silver.{repo_name}_pr_assignees
        WHERE assignee_id IS NOT NULL
        UNION
        SELECT DISTINCT
            user_id,
            user_login,
            user_type
        FROM silver.{repo_name}_pull_requests
        WHERE user_id IS NOT NULL
        UNION
        SELECT DISTINCT 
            reviewer_id AS user_id,
            reviewer_login AS user_login,
            reviewer_type AS user_type
        FROM silver.{repo_name}_pr_reviewers
        WHERE reviewer_id IS NOT NULL
        UNION
        SELECT DISTINCT
            author_id,
            author_login,
            author_type
        FROM silver.{repo_name}_commits
        WHERE author_id IS NOT NULL
        UNION
        SELECT DISTINCT
            committer_id,
            committer_login,
            committer_type
        FROM silver.{repo_name}_commits
        WHERE committer_id IS NOT NULL
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE VIEW gold.dim_user
    AS
    WITH combined_repos AS (
        {combined_subquery}
    ),
    null_users AS (
        SELECT 
            -1 AS user_key,
            NULL AS user_id,
            'n/a' AS user_login,
            'n/a' AS user_type    
    )
    SELECT DISTINCT
        ROW_NUMBER() OVER(ORDER BY user_id) AS user_key,
        user_id,
        user_login,
        user_type
    FROM combined_repos  
    UNION ALL
    SELECT * FROM null_users  
"""
spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.dim_user ORDER BY user_key

In [0]:
%sql
SELECT * FROM gold.dim_user WHERE user_id IS NULL OR user_login = 'n/a' OR user_type = 'n/a'

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT 
            pr_id,
            assignee_id
        FROM silver.{repo_name}_pr_assignees
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE VIEW gold.pr_assignee
    AS
    WITH combined_repos AS (
        {combined_subquery}
    )
    SELECT DISTINCT
        cr.pr_id,
        COALESCE(du.user_key, -1) AS assignee_key
    FROM combined_repos cr
    LEFT JOIN gold.dim_user du ON cr.assignee_id = du.user_id
"""
spark.sql(full_query)

In [0]:
%sql
WITH old_nulls AS (
    SELECT pr_id FROM silver.aspnetcore_pr_assignees WHERE assignee_id IS NULL
    UNION ALL
    SELECT pr_id FROM silver.powershell_pr_assignees WHERE assignee_id IS NULL
    UNION ALL
    SELECT pr_id FROM silver.powertoys_pr_assignees WHERE assignee_id IS NULL
    UNION ALL
    SELECT pr_id FROM silver.runtime_pr_assignees WHERE assignee_id IS NULL
    UNION ALL
    SELECT pr_id FROM silver.terminal_pr_assignees WHERE assignee_id IS NULL
    UNION ALL
    SELECT pr_id FROM silver.typescript_pr_assignees WHERE assignee_id IS NULL
    UNION ALL
    SELECT pr_id FROM silver.vscode_pr_assignees WHERE assignee_id IS NULL
),
new_negative_ones AS (
    SELECT pr_id FROM gold.pr_assignee WHERE assignee_key = -1
)
SELECT 
    (SELECT COUNT(*) FROM old_nulls) AS old_null_count,
    (SELECT COUNT(*) FROM new_negative_ones) AS new_neg1_count,
    (SELECT COUNT(*) FROM old_nulls o WHERE NOT EXISTS (SELECT 1 FROM new_negative_ones n WHERE n.pr_id = o.pr_id)) AS mismatched

In [0]:
%sql
WITH all_assignees AS (
    SELECT assignee_id FROM silver.aspnetcore_pr_assignees
    UNION ALL
    SELECT assignee_id FROM silver.powershell_pr_assignees
    UNION ALL
    SELECT assignee_id FROM silver.powertoys_pr_assignees
    UNION ALL
    SELECT assignee_id FROM silver.runtime_pr_assignees
    UNION ALL
    SELECT assignee_id FROM silver.terminal_pr_assignees
    UNION ALL
    SELECT assignee_id FROM silver.typescript_pr_assignees
    UNION ALL
    SELECT assignee_id FROM silver.vscode_pr_assignees
)
SELECT DISTINCT du.user_key, du.user_id, du.user_login
FROM all_assignees a
JOIN gold.dim_user du ON a.assignee_id = du.user_id
WHERE du.user_key BETWEEN 1 AND 5
ORDER BY du.user_key

In [0]:
%sql
SELECT * FROM gold.pr_assignee WHERE assignee_key != -1 ORDER BY assignee_key

In [0]:
%sql
SELECT * FROM silver.aspnetcore_pr_assignees WHERE assignee_id IS NULL

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT 
            pr_id,
            reviewer_id
        FROM silver.{repo_name}_pr_reviewers
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE VIEW gold.pr_reviewer
    AS
    WITH combined_repos AS (
        {combined_subquery}
    )
    SELECT DISTINCT
        cr.pr_id,
        COALESCE(du.user_key, -1) AS reviewer_key
    FROM combined_repos cr
    LEFT JOIN gold.dim_user du ON cr.reviewer_id = du.user_id
"""
spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.pr_reviewer WHERE reviewer_key != -1 ORDER BY reviewer_key

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT 
            label_id,
            label_name,
            label_color
        FROM silver.{repo_name}_pr_labels
        WHERE label_id IS NOT NULL
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ALL ".join(union_parts)

full_query = f"""
    CREATE OR REPLACE VIEW gold.dim_label
    AS
    WITH combined_repos AS (
        {combined_subquery}
    ),
    resolved_labels AS (
        SELECT
            label_id,
            MAX(label_name)  AS label_name,
            MAX(label_color) AS label_color
        FROM combined_repos
        GROUP BY label_id
    ),
    not_nulls AS (
        SELECT 
            ROW_NUMBER() OVER(ORDER BY label_id) AS label_key,
            label_id,
            label_name,
            label_color
        FROM resolved_labels
    ),
    null_label AS (
        SELECT -1 AS label_key, CAST(NULL AS BIGINT) AS label_id, 'n/a' AS label_name, 'n/a' AS label_color
    )
    SELECT * FROM not_nulls
    UNION ALL
    SELECT * FROM null_label   
"""
spark.sql(full_query)

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT DISTINCT
            label_id,
            label_name,
            label_color
        FROM silver.{repo_name}_pr_labels
        WHERE label_id IS NULL OR label_name = 'n/a' OR label_color = 'n/a'
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
df = spark.sql(combined_subquery)
display(df)

In [0]:
%sql
SELECT * FROM gold.dim_label ORDER BY label_key

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""SELECT 
            pr_id,
            label_id
        FROM silver.{repo_name}_pr_labels
    """
    for repo_name in repo_names
]
combined_subquery = " UNION ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE VIEW gold.pr_label
    AS
    WITH combined_repos AS (
        {combined_subquery}
    )
    SELECT DISTINCT
        cr.pr_id,
        COALESCE(dl.label_key, -1) AS label_key
    FROM combined_repos cr
    LEFT JOIN gold.dim_label dl ON cr.label_id = dl.label_id
"""
spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.pr_label WHERE label_key != -1 ORDER BY label_key

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]

union_parts = [
    f"""
    SELECT 
        pr.pr_id,
        pr.number,
        pr.merge_commit_sha,
        COALESCE(re.repo_key, -1) AS repo_key,
        COALESCE(u.user_key, -1) AS author_key,
        COALESCE(au.author_association_key, -1) AS author_association_key,
        COALESCE(f.flag_key, -1) AS flag_key,
        cd.date_key AS created_date_key,
        ct.timestamp_id AS created_timestamp_key,
        ud.date_key AS updated_date_key,
        ut.timestamp_id AS updated_timestamp_key,
        cld.date_key AS closed_date_key,
        clt.timestamp_id AS closed_timestamp_key,
        md.date_key AS merged_date_key,
        mt.timestamp_id AS merged_timestamp_key,
        COALESCE(br.base_ref_key, -1) AS base_ref_key,
        COALESCE(ms.milestone_key, -1) AS milestone_key,
        CASE 
            WHEN pr.base_repo_id IS NULL OR pr.head_repo_id IS NULL THEN NULL
            WHEN pr.base_repo_id != pr.head_repo_id THEN 1 
            ELSE 0 
        END AS is_external,
        TIMESTAMPDIFF(SECOND, pr.created_at, pr.merged_at) / 3600.0 AS time_to_merge_hours,
        COALESCE(pa.known_assignees_count, 0) AS assignees_count,
        COALESCE(rev.known_reviewers_count, 0) AS reviewers_count,
        COALESCE(pl.known_labels_count, 0) AS labels_count
    FROM silver.{repo_name}_pull_requests pr
    LEFT JOIN gold.dim_repo re ON pr.base_repo_id = re.repo_id
    LEFT JOIN gold.dim_user u ON pr.user_id = u.user_id 
    LEFT JOIN gold.dim_author_association au ON pr.author_association = au.author_association
    LEFT JOIN gold.dim_flags f ON COALESCE(pr.state, 'n/a') = f.state
        AND (CASE WHEN pr.draft = True THEN 'Draft' ELSE 'Not Draft' END) = f.draft_label
        AND (CASE WHEN pr.locked = True THEN 'Locked' ELSE 'Not Locked' END) = f.locked_label
        AND COALESCE(pr.auto_merge_method, 'n/a') = f.auto_merge_method
    LEFT JOIN gold.dim_date cd ON CAST(date_format(pr.created_at, 'yyyyMMdd') AS INT) = cd.date_key
    LEFT JOIN gold.dim_timestamp ct ON (HOUR(pr.created_at) * 60) + MINUTE(pr.created_at) = ct.timestamp_id
    LEFT JOIN gold.dim_date ud ON CAST(date_format(pr.updated_at, 'yyyyMMdd') AS INT) = ud.date_key
    LEFT JOIN gold.dim_timestamp ut ON (HOUR(pr.updated_at) * 60) + MINUTE(pr.updated_at) = ut.timestamp_id
    LEFT JOIN gold.dim_date cld ON CAST(date_format(pr.closed_at, 'yyyyMMdd') AS INT) = cld.date_key
    LEFT JOIN gold.dim_timestamp clt ON (HOUR(pr.closed_at) * 60) + MINUTE(pr.closed_at) = clt.timestamp_id
    LEFT JOIN gold.dim_date md ON CAST(date_format(pr.merged_at, 'yyyyMMdd') AS INT) = md.date_key
    LEFT JOIN gold.dim_timestamp mt ON (HOUR(pr.merged_at) * 60) + MINUTE(pr.merged_at) = mt.timestamp_id
    LEFT JOIN gold.dim_milestone ms ON pr.milestone_id = ms.milestone_id
    LEFT JOIN pa_agg pa ON pr.pr_id = pa.pr_id
    LEFT JOIN rev_agg rev ON pr.pr_id = rev.pr_id
    LEFT JOIN pl_agg pl ON pr.pr_id = pl.pr_id
    LEFT JOIN gold.dim_base_ref br 
    ON pr.base_repo_id = br.base_repo_id 
    AND pr.base_ref = br.base_ref
    """
    for repo_name in repo_names
]

combined_subquery = " UNION ALL ".join(union_parts)

full_query = f"""
CREATE OR REPLACE TABLE gold.fact_pull_request AS
WITH pa_agg AS (
    SELECT pr_id, COUNT(DISTINCT assignee_key) AS known_assignees_count 
    FROM gold.pr_assignee
    WHERE assignee_key != -1
    GROUP BY pr_id
),
rev_agg AS (
    SELECT pr_id, COUNT(DISTINCT reviewer_key) AS known_reviewers_count 
    FROM gold.pr_reviewer 
    WHERE reviewer_key != -1
    GROUP BY pr_id
),
pl_agg AS (
    SELECT pr_id, COUNT(DISTINCT label_key) AS known_labels_count 
    FROM gold.pr_label 
    WHERE label_key != -1
    GROUP BY pr_id

),
combined_repos AS (
    {combined_subquery}
)
SELECT * FROM combined_repos
"""

spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.fact_pull_request

In [0]:
%sql
  SELECT COUNT(*), COUNT(DISTINCT pr_id) FROM gold.fact_pull_request

In [0]:
repo_names = ["aspnetcore", "powershell", "powertoys", "runtime", "terminal", "typescript", "vscode"]
union_parts = [
    f"""
    SELECT
        c.sha,
        COALESCE(au.user_key, -1) AS author_key,
        COALESCE(cu.user_key, -1) AS committer_key,
        r.repo_key,
        COALESCE(cad.date_key, -1) AS commit_author_date_key,
        COALESCE(cat.timestamp_id, -1) AS commit_author_timestamp_id,
        COALESCE(ccd.date_key, -1) AS commit_commiter_date_key,
        COALESCE(cct.timestamp_id, -1) AS commit_commiter_timestamp_id,
        c.comment_count,
        TIMESTAMPDIFF(SECOND, c.commit_author_date, c.commit_committer_date) / 3600.0 AS time_to_commit_hours
    FROM silver.{repo_name}_commits c
    LEFT JOIN gold.dim_repo r ON c.repo_name = r.name
    LEFT JOIN gold.dim_user au ON c.author_id = au.user_id
    LEFT JOIN gold.dim_user cu ON c.committer_id = cu.user_id
    LEFT JOIN gold.dim_date cad ON CAST(date_format(c.commit_author_date, 'yyyyMMdd') AS INT) = cad.date_key
    LEFT JOIN gold.dim_timestamp cat ON (HOUR(c.commit_author_date) * 60) + MINUTE(c.commit_author_date) = cat.timestamp_id
    LEFT JOIN gold.dim_date ccd ON CAST(date_format(c.commit_committer_date, 'yyyyMMdd') AS INT) = ccd.date_key
    LEFT JOIN gold.dim_timestamp cct ON (HOUR(c.commit_committer_date) * 60) + MINUTE(c.commit_committer_date) = cct.timestamp_id
    """ 
    for repo_name in repo_names
]
combined_subquery = " UNION ALL ".join(union_parts)
full_query = f"""
    CREATE OR REPLACE TABLE gold.fact_commits AS
    {combined_subquery}
"""
spark.sql(full_query)

In [0]:
%sql
SELECT * FROM gold.fact_commits WHERE repo_key = -1 OR commit_author_date_key = -1 OR commit_author_timestamp_id = -1 OR commit_commiter_date_key = -1 OR commit_commiter_timestamp_id = -1 OR comment_count < 0 OR comment_count IS NULL OR time_to_commit_hours IS NULL OR time_to_commit_hours < 0

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT sha) FROM gold.fact_commits

In [0]:
%sql
SELECT base_ref, COUNT(*) 
FROM gold.dim_base_ref 
GROUP BY base_ref 
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT 'created_date_key' AS column_name, COUNT(*) AS rows_before_2024
FROM gold.fact_pull_request
WHERE created_date_key != -1 AND created_date_key < 20240101

UNION ALL

SELECT 'updated_date_key', COUNT(*)
FROM gold.fact_pull_request
WHERE updated_date_key != -1 AND updated_date_key < 20240101

UNION ALL

SELECT 'closed_date_key', COUNT(*)
FROM gold.fact_pull_request
WHERE closed_date_key != -1 AND closed_date_key < 20240101

UNION ALL

SELECT 'merged_date_key', COUNT(*)
FROM gold.fact_pull_request
WHERE merged_date_key != -1 AND merged_date_key < 20240101

ORDER BY rows_before_2024 DESC;